# T04 — ConvNeXt-Tiny: CutMix alpha=1.0; target/loss theo lambda thực tế
**Phần 8 · 2A202602810 Nguyễn Thái Anh**

Nền T00 = B03/seed0. Giữ 12 epoch, batch 64, ảnh 224, LR và scheduler.
Notebook này chạy một thí nghiệm. Có thể dùng các notebook riêng thay cho master;
chỉ có một process cho mỗi ID. Chọn kernel `.venv` và Run All, tự resume/đọc run hoàn tất.

## 1. Môi trường

In [ ]:
import os, sys, json
from pathlib import Path
from dataclasses import asdict
ROOT_HINT = None
ROOT = Path(ROOT_HINT).expanduser().resolve() if ROOT_HINT else next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (p / "eval.py").is_file() and (p / "starter").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Đặt ROOT_HINT tới gốc repo")
CODE = ROOT / "submissions/2A202602810_NguyenThaiAnh/code"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HOME"] = str(ROOT / ".cache/huggingface")
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache/matplotlib"))
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(CODE))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown
import ablation_notebook as ab
import backbone_notebook as nbtrain
import train as trainer
assert Path(trainer.__file__).resolve() == CODE / "train.py"
print("ROOT:", ROOT)
print("Python:", sys.executable)

## 2. Chọn run và kiểm tra nền

In [ ]:
EXP_ID = "T04"
RUN_TRAINING = True

plan = ab.active_plan()
configs = ab.configs_for(plan)
base = configs["T00"]
summary, baseline_history, metrics = ab.verify_ablation(base, base)
print(f"T00 = B03/seed{base.seed}; F1 val = {metrics['macro_f1']:.4%}; đủ {base.epochs} epoch")
display(pd.DataFrame([{"exp_id":s.exp_id, "axis":s.axis, "changes":s.changes,
                       "status":nbtrain.training_status(configs[s.exp_id])} for s in plan]))
print("Không tái train B03; mọi run T kế thừa seed/split/batch/LR/scheduler/preprocessing của B03.")

cfg = configs[EXP_ID]
display(pd.DataFrame([asdict(cfg)]).T)

## 3. Train và audit

In [ ]:
if RUN_TRAINING:
    nbtrain.run_in_subprocess(cfg, planned_group=4)
if nbtrain.training_status(cfg) == "complete":
    result, history, metrics = ab.verify_ablation(cfg, base)
    display(result)
    display(history)
    display(Image(filename=str(Path(cfg.curves_dir) / f"{cfg.exp_id}_seed{cfg.seed}_{cfg.backbone}.png")))
    print("Output audit: PASS")
else:
    print("Chưa hoàn thành run; bật RUN_TRAINING và chạy trên host có GPU.")

## 4. Bước tiếp theo
Sau khi T01–T07 hoàn tất, dùng `train_T08_convnext_combination.ipynb` hoặc master để chọn/run tổ hợp,
và tổng hợp bảng/F1 theo lớp. Notebook riêng không ghi bảng tổng dùng chung giữa các job.